# Falsh Attention 原理与代码

Flash Attention 是 Tri Dao 等人在 2022 年提出的一种精确(exact)注意力计算算法,核心目标是让 Transformer 的注意力计算更快、更省显存。它现在几乎是所有主流大模型训练和推理的标配。下面我分几个部分介绍。

它要解决的问题
标准注意力的计算是这三步:
S = Q · Kᵀ        (注意力分数,N×N 矩阵)
P = softmax(S)     (N×N)
O = P · V          (输出)



 其中 N 是序列长度。问题在于那个 N×N 的中间矩阵:序列越长,它越大,显存占用是 O(N²)。更关键的是,真正的瓶颈不是算力(FLOPs),而是显存读写(IO)——这个大矩阵要反复在 GPU 的全局显存(HBM)里读出、写入,数据搬运的时间远超实际计算时间。换句话说,标准注意力是 memory-bound(访存受限) 而非 compute-bound。

这一点是理解 Flash Attention 的关键:它本质上是个 IO 优化算法,而不是一个数学近似。
下面这张图展示了 GPU 的内存层级和它的核心思路:

Flash Attention 的核心洞察来自 GPU 的内存层级:芯片上的 SRAM 极快但很小,而全局显存 HBM 很大但慢得多。标准注意力把那个 N×N 大矩阵反复写进 HBM,慢就慢在这里。Flash Attention 的做法是把数据切成小块,只在快速的 SRAM 里完成计算:

![N×N 分数矩阵在高速 SRAM 中算完,不写回慢速 HBM](/home/orange/orangeai/LLM/EasyLLM/asserts/images/flash_attention.png "N×N 分数矩阵在高速 SRAM 中算完,不写回慢速 HBM") 


三个关键技巧
1. 分块(Tiling)
把 Q、K、V 沿序列维度切成小块,每次只把一小块搬进 SRAM 计算,因此完整的 N×N 矩阵从不在 HBM 里出现。显存占用从 O(N²) 降到 O(N)(线性)。
2. 在线 softmax(Online Softmax)
分块的难点在于:softmax 需要对一整行做归一化,但你一次只看到一块。Flash Attention 用"在线 softmax"解决这个问题——一边遍历各块,一边维护两个运行统计量:当前最大值 m 和指数和 l。每来一个新块,就更新最大值,并按 exp(旧max − 新max) 把已累积的结果重新缩放。这样不必看到整行也能算出正确的 softmax,结果与标准注意力完全一致(不是近似)。
3. 反向传播时重算(Recomputation)
反向传播本来需要那个 N×N 矩阵。Flash Attention 不存它,而是利用前向保存的统计量在反向时重新算一遍。用少量额外计算换取大幅的显存节省——在访存受限的场景下,这笔交易非常划算。

效果
显存从 O(N²) 降到 O(N),让长序列(几千甚至几万 token)变得可行;墙钟时间通常有 2–4 倍加速,主要来自减少了 HBM 读写而非减少计算量;并且结果精确,不损失任何精度

In [1]:
"""
Desc:
Flash Attention 的简洁 PyTorch 实现(教学用)。  
说明:
- 这是用纯 PyTorch 模拟 Flash Attention 的「算法逻辑」,便于理解原理;
  真正的高性能版本是 CUDA / Triton 编写的融合 kernel(在 SRAM 上做分块计算)。
- 核心思想:分块(tiling) + 在线 softmax(online softmax),
  从不一次性构造完整的 N×N 注意力矩阵,因此显存占用是 O(N) 而非 O(N²),
  且结果与标准注意力「完全一致」(精确,不是近似)。
- 为保持简洁,这里只对 K/V 沿序列维度分块,Q 一次性放入(相当于常驻 SRAM)。
  完整版本会同时对 Q 也分块(双重循环)。

Author: Orange
Date: 2026/06/02
"""

import torch


def flash_attention(Q, K, V, block_size=128):
    """
    参数:
        Q: 查询矩阵,形状 (N, d)
        K: 键矩阵,  形状 (N, d)
        V: 值矩阵,  形状 (N, d)
        block_size: K/V 的分块大小(对应每次搬进「SRAM」的块)
    返回:
        O: 输出矩阵,形状 (N, d)
    """
    N, d = Q.shape
    scale = 1.0 / (d ** 0.5)  # 缩放因子 1/sqrt(d),与标准注意力一致

    # 输出累加器 O,以及每一行的两个「运行统计量」
    O = torch.zeros_like(Q)                       # 输出 O,形状 (N, d)
    row_max = torch.full((N,), float("-inf"))     # 每行运行最大值 m,初始为 -inf
    row_sum = torch.zeros(N)                       # 每行运行指数和 l,初始为 0

    # 外层循环:逐块遍历 K / V(每次只处理一个块,而非整个矩阵)
    for j in range(0, N, block_size):
        K_block = K[j:j + block_size]   # 当前 K 块,形状 (Bk, d)
        V_block = V[j:j + block_size]   # 当前 V 块,形状 (Bk, d)

        # 1) 计算 Q 与当前 K 块的注意力分数 S = Q · Kᵀ · scale,形状 (N, Bk)
        S = (Q @ K_block.T) * scale

        # 2) 在线 softmax:用「运行统计量」增量地完成归一化 ----------------
        block_max = S.max(dim=1).values               # 当前块每行的最大值,(N,)
        new_max = torch.maximum(row_max, block_max)    # 更新后的运行最大值,(N,)

        # 旧统计量(已累积的 O 和 row_sum)需要按新最大值重新缩放
        # 缩放系数 = exp(旧max − 新max);首次时 exp(-inf − 新max) = 0,自动清零初值
        correction = torch.exp(row_max - new_max)      # (N,)

        # 当前块的(未归一化)概率,减去 new_max 保证数值稳定
        P = torch.exp(S - new_max[:, None])            # (N, Bk)

        # 更新运行指数和:旧和缩放后 + 当前块的和
        row_sum = row_sum * correction + P.sum(dim=1)

        # 更新输出:旧输出缩放后 + 当前块的贡献 (P · V_block)
        O = O * correction[:, None] + P @ V_block

        # 保存新的运行最大值,供下一块使用
        row_max = new_max

    # 3) 全部块遍历完后,用累计的指数和做最终归一化
    O = O / row_sum[:, None]
    return O


def naive_attention(Q, K, V):
    """标准注意力(参考实现),会显式构造完整的 N×N 矩阵。"""
    scale = 1.0 / (Q.shape[-1] ** 0.5)
    S = (Q @ K.T) * scale          # 完整的 N×N 分数矩阵(显存 O(N²))
    P = torch.softmax(S, dim=-1)   # 整行 softmax
    return P @ V


if __name__ == "__main__":
    torch.manual_seed(0)
    N, d = 512, 64
    Q = torch.randn(N, d)
    K = torch.randn(N, d)
    V = torch.randn(N, d)

    out_flash = flash_attention(Q, K, V, block_size=128)
    out_naive = naive_attention(Q, K, V)

    # 两者应当几乎完全相等(误差仅来自浮点累加顺序)
    print("最大绝对误差:", (out_flash - out_naive).abs().max().item())
    print("数值是否一致:", torch.allclose(out_flash, out_naive, atol=1e-5))

最大绝对误差: 2.086162567138672e-07
数值是否一致: True
